# Week 8 Pair Programming: Architecture Tour

## Goal
Get hands-on intuition for how CNN architecture choices affect accuracy and parameter count.

## Setup
Find a partner. One of you drives (writes code), the other navigates (reads, asks questions, suggests changes). **Switch roles every 5 minutes.**

## Time
25 minutes total (5 min × 5 variants, with role swaps).

## What you'll do
Train 5 CNN variants on Fashion-MNIST and fill in a comparison table. The baseline is the CNN you built in the live session.

## Continuity
This builds directly on `week8_live_session.ipynb`. The data loading and training pattern are identical.

## Setup (run this first)

In [1]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

keras.utils.set_random_seed(42)

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train = (X_train.astype('float32') / 255.0)[..., np.newaxis]
X_test = (X_test.astype('float32') / 255.0)[..., np.newaxis]

# Use a subset for speed during pair programming
X_train_sub, y_train_sub = X_train[:20000], y_train[:20000]

print(f'Training subset shape: {X_train_sub.shape}')
print(f'Test set shape: {X_test.shape}')

Training subset shape: (20000, 28, 28, 1)
Test set shape: (10000, 28, 28, 1)


## Helper: train and evaluate any model

We've abstracted the train/eval logic so you can focus on the architecture changes.

In [2]:
def train_and_eval(model, name, epochs=5):
    """Train a model on the Fashion-MNIST subset and report stats."""
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(
        X_train_sub, y_train_sub,
        epochs=epochs, batch_size=128, validation_split=0.1, verbose=0
    )
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    n_params = model.count_params()
    print(f'{name:30s} | params: {n_params:>9,} | test_acc: {test_acc:.4f}')
    return {
        'name': name, 'params': n_params, 'test_acc': test_acc,
        'history': history.history
    }

results = []

## Variant 1: Baseline (the live session CNN)

**Driver:** run this cell. **Navigator:** note the parameter count and test accuracy on the comparison table at the bottom.

**Time:** 5 minutes (most of which is training time — discuss with partner while it runs).

In [3]:
baseline = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='baseline')

results.append(train_and_eval(baseline, 'Variant 1: Baseline'))

Variant 1: Baseline            | params:   110,634 | test_acc: 0.8647


## Variant 2: Deeper (3 conv blocks instead of 2)

**Switch roles.** New driver implements this variant.

**Hypothesis to discuss:** Does adding a third Conv→Pool block help or hurt? Why might it not help here?

In [4]:
deeper = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),
    layers.Conv2D(32, 3, activation='relu', padding='same'),  # NEW: third conv block
    layers.MaxPooling2D(2),                                    # NEW: third pool (7x7 -> 3x3)
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='deeper')

results.append(train_and_eval(deeper, 'Variant 2: Deeper'))

Variant 2: Deeper              | params:    37,962 | test_acc: 0.8427


## Variant 3: Wider (more filters per layer)

**Switch roles.**

**Hypothesis:** What does doubling the number of filters do to parameter count? To accuracy?

In [5]:
wider = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(64, 3, activation='relu', padding='same'),   # 32 -> 64 filters
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu', padding='same'),   # 32 -> 64 filters
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='wider')

results.append(train_and_eval(wider, 'Variant 3: Wider'))

Variant 3: Wider               | params:   238,986 | test_acc: 0.8735


## Variant 4: Larger kernel size (5×5 instead of 3×3)

**Switch roles.**

**Hypothesis:** Larger kernels see more spatial context per filter. Does that help?

In [6]:
larger_kernel = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 5, activation='relu', padding='same'),   # 3 -> 5
    layers.MaxPooling2D(2),
    layers.Conv2D(32, 5, activation='relu', padding='same'),   # 3 -> 5
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='larger_kernel')

results.append(train_and_eval(larger_kernel, 'Variant 4: 5x5 kernels'))

Variant 4: 5x5 kernels         | params:   127,530 | test_acc: 0.8718


## Variant 5: AveragePooling instead of MaxPooling

**Switch roles.**

**Hypothesis:** MaxPooling preserves the strongest activation; AveragePooling smooths. Which works better here? Why?

In [7]:
avg_pool = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.AveragePooling2D(2),  # MaxPooling -> AveragePooling
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.AveragePooling2D(2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='avg_pool')

results.append(train_and_eval(avg_pool, 'Variant 5: AveragePooling'))

Variant 5: AveragePooling      | params:   110,634 | test_acc: 0.8469


## Comparison summary

In [8]:
print(f'{"Variant":35s} | {"Params":>10s} | {"Test Acc":>10s}')
print('-' * 65)
for r in results:
    print(f'{r["name"]:35s} | {r["params"]:>10,} | {r["test_acc"]:>10.4f}')

# Identify the winner
winner = max(results, key=lambda r: r['test_acc'])
print(f'\nWinner: {winner["name"]} ({winner["test_acc"]:.4f})')

Variant                             |     Params |   Test Acc
-----------------------------------------------------------------
Variant 1: Baseline                 |    110,634 |     0.8647
Variant 2: Deeper                   |     37,962 |     0.8427
Variant 3: Wider                    |    238,986 |     0.8735
Variant 4: 5x5 kernels              |    127,530 |     0.8718
Variant 5: AveragePooling           |    110,634 |     0.8469

Winner: Variant 3: Wider (0.8735)


## Discussion (with your partner — 5 minutes)

**Answer these together:**

1. **Did deeper always mean better?** Why might a third conv block on a 28×28 image not help?
2. **Wider added a lot of parameters.** Did it improve accuracy proportionally?
3. **Did the larger kernel (5×5) help?** What's the tradeoff (parameters per filter, receptive field)?
4. **Max vs Average pooling** — which won, and what does that say about Fashion-MNIST images?
5. **If you had to pick ONE variant for production**, which would it be? Consider both accuracy AND parameter efficiency.

## Solutions / typical observations

(Don't peek until you've discussed.)

<details>
<summary>Click to reveal typical observations</summary>

- **Deeper:** With 28×28 input, a third pool reduces spatial size to 3×3 — too small to be useful. Deeper helps for larger images, not so much here.
- **Wider:** Tends to help accuracy slightly but adds significant params. Diminishing returns.
- **5×5 kernels:** Often comparable to 3×3, but with more parameters per filter. Modern practice is to stack 3×3s instead (two 3×3s have receptive field equivalent to one 5×5 with fewer parameters).
- **Average pooling:** Usually slightly worse than max pooling on classification. Max preserves the strongest signal; average smooths. For object recognition, you usually care about whether a strong pattern is present (max).
- **Production pick:** Baseline often wins on the parameters-per-accuracy tradeoff. Don't assume more is better.
</details>

## Bonus (if time permits)

Try ONE more variant of YOUR design. Predict the result first, then run it.

Ideas:
- BatchNormalization layer after each Conv2D
- Two Dense layers in the head instead of one
- GlobalAveragePooling2D instead of Flatten + Dense
- Stride 2 conv instead of MaxPool

Write your prediction in a markdown cell, then implement, then compare to your prediction.

In [9]:
# Your bonus variant goes here
# Prediction: ...

my_variant = keras.Sequential([
    # TODO: design your own
])